# Creating Custom Bias

## Introduction

In NSGABLACK, custom bias allows you to embed domain-specific knowledge and optimization strategies. This tutorial will show you how to create and integrate custom bias.

## Basic Structure of Custom Bias

All bias classes should inherit from the base class `BiasBase`:

In [ ]:
# Import base classes
from bias.bias_base import BiasBase
import numpy as np

# View the base class structure
print(BiasBase.__doc__)

In [ ]:
# Create a simple custom bias
class CustomBias(BiasBase):
    """Example of a simple custom bias
    
    This bias will guide solutions toward lower values
    """
    def __init__(self, name="CustomBias", strength=0.1, target_value=0.0):
        super().__init__(name=name)
        self.strength = strength
        self.target_value = target_value
        self.generation = 0
        
    def apply(self, population, problem=None, generation=0):
        """Apply bias to the population
        
        Args:
            population: List of individuals
            problem: Optimization problem (optional)
            generation: Current generation
            
        Returns:
            List of biased individuals
        """
        self.generation = generation
        biased_population = []
        
        for individual in population:
            biased_individual = self._bias_individual(individual)
            biased_population.append(biased_individual)
            
        return biased_population
    
    def _bias_individual(self, individual):
        """Apply bias to a single individual"""
        if isinstance(individual, np.ndarray):
            # Pull individual values toward target
            direction = self.target_value - individual
            biased = individual + direction * self.strength
            return biased
        else:
            # For other types, return as is
            return individual

# Test the custom bias
bias = CustomBias(strength=0.2)
test_pop = [np.array([1.0, 2.0, 3.0]), np.array([0.5, 1.5, 2.5])]
biased_pop = bias.apply(test_pop)

print("Original population:", test_pop)
print("Biased population:", biased_pop)

## Example 1: Guiding Bias for TSP Problems

Create a bias for the Traveling Salesman Problem that uses city distance information:

In [ ]:
class TSPGuidedBias(BiasBase):
    """Bias for TSP problems: uses city distances to guide search"""
    
    def __init__(self, distance_matrix, strength=0.1):
        super().__init__(name="TSP_Guided_Bias")
        self.distance_matrix = np.array(distance_matrix)
        self.strength = strength
        self.n_cities = len(distance_matrix)
        
        # Pre-calculate nearest neighbors for each city
        self.nearest_neighbors = {}
        for i in range(self.n_cities):
            neighbors = np.argsort(self.distance_matrix[i])[1:4]  # 3 nearest
            self.nearest_neighbors[i] = neighbors
    
    def apply(self, population, problem=None, generation=0):
        biased_population = []
        
        for individual in population:
            if isinstance(individual, list) and len(individual) == self.n_cities:
                # Individual is a city tour
                biased_tour = self._improve_tour(individual)
                biased_population.append(biased_tour)
            else:
                biased_population.append(individual)
                
        return biased_population
    
    def _improve_tour(self, tour):
        """Use 2-opt to improve the tour"""
        tour = tour.copy()
        improved = True
        
        while improved and np.random.rand() < self.strength:
            improved = False
            for i in range(len(tour) - 2):
                for j in range(i + 2, len(tour)):
                    # Calculate current distance
                    current_dist = (self.distance_matrix[tour[i-1]][tour[i]] +
                                   self.distance_matrix[tour[j-1]][tour[j]])
                    
                    # Calculate distance after reversal
                    new_dist = (self.distance_matrix[tour[i-1]][tour[j-1]] +
                               self.distance_matrix[tour[i]][tour[j]])
                    
                    # If improvement found, apply 2-opt
                    if new_dist < current_dist:
                        tour[i:j] = tour[i:j][::-1]
                        improved = True
                        break
                if improved:
                    break
                    
        return tour

# Create a simple distance matrix for testing
dist_matrix = [
    [0, 2, 3, 4, 5],
    [2, 0, 4, 3, 2],
    [3, 4, 0, 1, 3],
    [4, 3, 1, 0, 2],
    [5, 2, 3, 2, 0]
]

# Create TSP bias
tsp_bias = TSPGuidedBias(dist_matrix, strength=0.3)

# Test bias effect
test_tour = [0, 2, 1, 3, 4]  # A possible tour
biased_tour = tsp_bias.apply([test_tour])[0]

print("Original tour:", test_tour)
print("Biased tour:", biased_tour)

## Example 2: Adaptive Bias

Create a bias that automatically adjusts based on optimization progress:

In [ ]:
class AdaptiveStrengthBias(BiasBase):
    """Bias with adaptive strength adjustment"""
    
    def __init__(self, initial_strength=0.2, min_strength=0.01, max_strength=0.5):
        super().__init__(name="Adaptive_Strength_Bias")
        self.strength = initial_strength
        self.min_strength = min_strength
        self.max_strength = max_strength
        self.history = []
        self.last_improvement = 0
        self.stagnation_threshold = 10
    
    def apply(self, population, problem=None, generation=0):
        # Update strength based on convergence situation
        self._update_strength(population, generation)
        
        # Apply bias
        biased_population = []
        for individual in population:
            biased = self._apply_bias(individual)
            biased_population.append(biased)
            
        return biased_population
    
    def _update_strength(self, population, generation):
        """Update bias strength based on population state"""
        # Calculate fitness values (using simple sphere function as example)
        fitness_values = [self._evaluate_fitness(ind) for ind in population]
        best_fitness = min(fitness_values)
        
        # Record history
        self.history.append(best_fitness)
        
        # Check for improvement
        if len(self.history) > 1:
            if best_fitness < self.history[-2]:
                self.last_improvement = generation
        
        # Adjust strength
        generations_since_improvement = generation - self.last_improvement
        
        if generations_since_improvement > self.stagnation_threshold:
            # Stagnation: increase strength
            self.strength = min(self.max_strength, self.strength * 1.1)
        else:
            # Improving: reduce strength
            self.strength = max(self.min_strength, self.strength * 0.95)
    
    def _evaluate_fitness(self, individual):
        """Simple fitness evaluation"""
        if isinstance(individual, np.ndarray):
            return np.sum(individual**2)
        return float('inf')
    
    def _apply_bias(self, individual):
        """Apply bias to a single individual"""
        if isinstance(individual, np.ndarray):
            # Random perturbation with current strength
            noise = np.random.randn(*individual.shape) * self.strength
            return individual + noise
        return individual

# Test adaptive bias
adaptive_bias = AdaptiveStrengthBias()
pop = [np.random.randn(5) for _ in range(20)]

strength_history = []
for gen in range(50):
    adaptive_bias.apply(pop, generation=gen)
    strength_history.append(adaptive_bias.strength)
    # Simulate evolution
    pop = [ind * 0.95 + np.random.randn(5) * 0.01 for ind in pop]

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.plot(strength_history)
plt.xlabel('Generation')
plt.ylabel('Bias Strength')
plt.title('Adaptive Bias Strength Adjustment')
plt.grid(True)
plt.show()

## Example 3: Composite Bias

Create a bias that combines multiple strategies:

In [ ]:
class CompositeBias(BiasBase):
    """Composite bias that combines multiple bias strategies"""
    
    def __init__(self, biases, weights=None):
        super().__init__(name="Composite_Bias")
        self.biases = biases
        
        # If weights not provided, use equal weights
        if weights is None:
            self.weights = [1.0 / len(biases)] * len(biases)
        else:
            # Normalize weights
            total = sum(weights)
            self.weights = [w / total for w in weights]
    
    def apply(self, population, problem=None, generation=0):
        # Apply each bias and combine results
        biased_populations = []
        
        for bias in self.biases:
            biased_pop = bias.apply(population, problem, generation)
            biased_populations.append(biased_pop)
        
        # Combine results based on weights
        final_population = []
        for i in range(len(population)):
            combined_individual = None
            
            for j, biased_pop in enumerate(biased_populations):
                if combined_individual is None:
                    combined_individual = biased_pop[i] * self.weights[j]
                else:
                    combined_individual += biased_pop[i] * self.weights[j]
            
            final_population.append(combined_individual)
        
        return final_population

# Create multiple simple biases
bias1 = CustomBias(name="Bias_To_Zero", strength=0.1, target_value=0.0)
bias2 = CustomBias(name="Bias_To_One", strength=0.1, target_value=1.0)

# Create composite bias
composite = CompositeBias([bias1, bias2], weights=[0.7, 0.3])

# Test composite bias
test_pop = [np.array([0.5, 0.5]), np.array([0.8, 0.8])]
biased_pop = composite.apply(test_pop)

print("Original population:", test_pop)
print("Biased population (70% to zero, 30% to one):", biased_pop)

## Integrating Custom Bias

### Method 1: Using in the solver

In [ ]:
from core.solver import NSGABLACKSolver

# Create custom bias
my_custom_bias = CustomBias(name="My_Bias", strength=0.15)

# Use bias in the solver
solver = NSGABLACKSolver(
    bias_system=my_custom_bias,
    algorithm="nsga2"
)

# Run optimization
# result = solver.optimize(...)

### Method 2: Adding to the bias library

In [ ]:
from bias.bias_library_algorithmic import AlgorithmicBiasLibrary

# Add custom bias to the library
bias_library = AlgorithmicBiasLibrary()
bias_library.register_bias("my_custom", CustomBias)

# Use through the library
bias_instance = bias_library.create_bias("my_custom", strength=0.2)

## Best Practices

### 1. Design Principles
- **Simplicity**: Avoid overly complex bias logic
- **Efficiency**: Bias calculation should not be too costly
- **Adaptability**: Should adapt to different optimization stages

### 2. Performance Considerations
- Cache computation results
- Avoid repeated calculations
- Use vectorized operations

### 3. Testing
```python
# Simple test template
def test_custom_bias():
    bias = CustomBias()
    test_pop = [np.random.rand(10) for _ in range(5)]
    biased_pop = bias.apply(test_pop)
    
    # Basic validation
    assert len(biased_pop) == len(test_pop)
    assert all(isinstance(ind, np.ndarray) for ind in biased_pop)
    print("Test passed!")
```

## Summary

Custom bias is a powerful mechanism that allows you to:

1. **Embed domain knowledge**: Incorporate problem-specific information
2. **Control search direction**: Guide algorithm toward promising regions
3. **Improve performance**: Accelerate convergence and solution quality

Key points:
- Inherit from `BiasBase` base class
- Implement the `apply` method
- Consider performance optimization
- Test thoroughly

## Next Tutorial

In the next tutorial, we'll learn about [Monte Carlo Optimization with Bias](03_Monte_Carlo_Algorithm_with_Bias.ipynb).